# 04 — Análise Temporal

**Objetivo:** colapsar o DataFrame granular (uma linha por CISP+mês+crime) numa série temporal mensal, pronta para plotar.
**Entrada:** DataFrame de `gerar_metricas()` — formato long com `taxa_100k`, `media_movel_3`, `variacao_pct`.
**Saída:** série temporal indexada por `mes_ano`, com totais e médias agregadas.
**Próximo passo:** copiar o consolidado para `analise_temporal()` em `pipeline.py`.

## Célula 1 — Setup

Aqui precisamos rodar todo o pipeline até `gerar_metricas()` para ter o dado certo.
Como as funções anteriores já estão implementadas, basta encadear as chamadas.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ))

from pipeline import carregar_dados, limpar_dados, gerar_metricas

ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

df = gerar_metricas(limpar_dados(carregar_dados(ARQUIVO)))
print("Entrada:", df.shape)
df.head(3)

2026-05-11 19:42:12 [INFO] Dados carregados: 37588 linhas, 65 colunas
2026-05-11 19:42:12 [INFO] Limpeza concluida. Shape final: (150352, 8)
2026-05-11 19:42:13 [INFO] Métricas geradas. Shape final: (150352, 12)


Entrada: (150352, 12)


,cisp,mes_ano,aisp,risp,munic,regiao,tipo_crime,qtd_ocorrencias,populacao,taxa_100k,media_movel_3,variacao_pct
75179,6,2003-01,1,1,RIO DE JANEIRO,CAPITAL,cvli,6,232000.0,2.586207,6.000000,NaN
75180,7,2003-01,1,1,RIO DE JANEIRO,CAPITAL,cvli,4,232000.0,1.724138,5.000000,-33.333333
75305,6,2003-02,1,1,RIO DE JANEIRO,CAPITAL,cvli,6,232000.0,2.586207,5.333333,50.000000


## Célula 2 — Entender o problema da granularidade

O DataFrame tem uma linha por CISP + mês + crime. Para plotar uma série temporal
precisamos de uma linha por mês — somando todas as CISPs e, opcionalmente, filtrando
por tipo de crime ou AISP.

Antes de agregar, vale ver quantas linhas existem por mês para entender o que
estamos colapsando.

In [2]:
linhas_por_mes = df.groupby("mes_ano").size()
print("Linhas por mês (primeiros 5):")
print(linhas_por_mes.head())
print()
print("Total de meses únicos:", df["mes_ano"].nunique())

Linhas por mês (primeiros 5):
mes_ano
2003-01    504
2003-02    504
2003-03    504
2003-04    508
2003-05    508
Freq: M, dtype: int64

Total de meses únicos: 279


## Célula 3 — Filtros condicionais

A função recebe `tipo_crime` e `aisp` como parâmetros opcionais.
O padrão `None` significa "sem filtro" — agrega tudo.

O `if tipo_crime:` só aplica o filtro quando o valor for passado.
Usar `.upper()` garante que `"hom_doloso"` e `"HOM_DOLOSO"` funcionem igual,
já que os dados foram normalizados para maiúsculas na limpeza.

In [9]:
# Experimente trocar esses valores para ver o resultado mudar
tipo_crime = None   # None = todos os crimes
aisp       = None           # None = todas as AISPs

df_filtrado = df.copy()

if tipo_crime:
    df_filtrado = df_filtrado[df_filtrado["tipo_crime"] == tipo_crime.lower()]

if aisp:
    df_filtrado = df_filtrado[df_filtrado["aisp"] == aisp.upper()]

print(f"Filtros aplicados — tipo_crime={tipo_crime!r}, aisp={aisp!r}")
print("Linhas após filtro:", df_filtrado.shape[0])

Filtros aplicados — tipo_crime=None, aisp=None
Linhas após filtro: 150352


## Célula 4 — Agregação mensal

Com o DataFrame filtrado, agrupamos por `mes_ano` e calculamos:
- `total_ocorrencias`: soma de todas as CISPs naquele mês — o número absoluto
- `taxa_100k_media`: média das taxas por 100k — o indicador normalizado
- `media_movel_3`: média da coluna já calculada (para manter consistência)

`agg()` permite aplicar funções diferentes em cada coluna de uma vez.
`.sort_index()` garante que o resultado fique em ordem cronológica.

In [10]:
serie = (
    df_filtrado
    .groupby("mes_ano")
    .agg(
        total_ocorrencias=("qtd_ocorrencias", "sum"),
        taxa_100k_media=("taxa_100k", "mean"),
        media_movel_3=("media_movel_3", "mean"),
    )
    .sort_index()
)

print("Shape da série:", serie.shape)
serie.head(6)

Shape da série: (279, 3)


,total_ocorrencias,taxa_100k_media,media_movel_3
mes_ano,,,
2003-01,1929,1.570033,3.651455
2003-02,1904,1.555289,3.822421
2003-03,2010,1.648149,4.016534
2003-04,1944,1.614620,3.997047
2003-05,1965,1.580805,3.887139
2003-06,1770,1.427805,3.705381


## Célula 5 — Inspecionar a série temporal

Com a série pronta, vale conferir se ela conta uma história coerente:
valores positivos, sem lacunas de meses, e totais que fazem sentido
para o tipo de crime filtrado.

In [11]:
print("Período:", serie.index.min(), "→", serie.index.max())
print("Meses na série:", len(serie))
print()
print("Estatísticas de total_ocorrencias:")
print(serie["total_ocorrencias"].describe().round(1))
print()
print("Mês com mais ocorrências:")
print(serie["total_ocorrencias"].idxmax(), "→", serie["total_ocorrencias"].max())
print()
print("Mês com menos ocorrências:")
print(serie["total_ocorrencias"].idxmin(), "→", serie["total_ocorrencias"].min())

Período: 2003-01 → 2026-03
Meses na série: 279

Estatísticas de total_ocorrencias:
count     279.0
mean     1308.5
std       350.7
min       613.0
25%      1017.5
50%      1275.0
75%      1605.5
max      2222.0
Name: total_ocorrencias, dtype: float64

Mês com mais ocorrências:
2005-03 → 2222

Mês com menos ocorrências:
2025-07 → 613


## Célula 6 — Validação: testar os três modos de uso

A função tem quatro combinações possíveis de filtro. Aqui testamos as principais
para garantir que todas retornam resultados válidos e com shapes coerentes.

Número de meses deve ser igual nos quatro casos — o filtro reduz colunas,
nunca remove períodos inteiros do histórico.

In [12]:
def agregar(df_in, tipo_crime=None, aisp=None):
    d = df_in.copy()
    if tipo_crime:
        d = d[d["tipo_crime"] == tipo_crime.lower()]
    if aisp:
        d = d[d["aisp"] == aisp.upper()]
    return (
        d.groupby("mes_ano")
         .agg(
             total_ocorrencias=("qtd_ocorrencias", "sum"),
             taxa_100k_media=("taxa_100k", "mean"),
             media_movel_3=("media_movel_3", "mean"),
         )
         .sort_index()
    )

casos = [
    ("Sem filtro",              None,           None),
    ("Só tipo_crime",           "hom_doloso",   None),
    ("Só aisp",                 None,           "1"),
    ("tipo_crime + aisp",       "hom_doloso",   "1"),
]

for label, tc, a in casos:
    s = agregar(df, tc, a)
    print(f"{label:25s} → {len(s)} meses, total={s['total_ocorrencias'].sum():,}")

Sem filtro                → 279 meses, total=365,065
Só tipo_crime             → 279 meses, total=109,621
Só aisp                   → 105 meses, total=2,335
tipo_crime + aisp         → 105 meses, total=654


---
## Consolidado — o que vai para `analise_temporal()` no pipeline.py

```python
def analise_temporal(
    df: pd.DataFrame,
    tipo_crime: str | None = None,
    aisp: str | None = None,
) -> pd.DataFrame:
    df = df.copy()

    if tipo_crime:
        df = df[df["tipo_crime"] == tipo_crime.lower()]

    if aisp:
        df = df[df["aisp"] == aisp.upper()]

    serie = (
        df.groupby("mes_ano")
          .agg(
              total_ocorrencias=("qtd_ocorrencias", "sum"),
              taxa_100k_media=("taxa_100k", "mean"),
              media_movel_3=("media_movel_3", "mean"),
          )
          .sort_index()
    )

    log.info(
        "Serie temporal: %s meses | crime=%s | aisp=%s",
        len(serie), tipo_crime or "todos", aisp or "todas",
    )
    return serie
```